# ERB · Kaggle runner (free T4 x2)
1. Locally: `python scripts/make_kaggle_bundle.py` → `erb2_bundle.zip`.
2. Kaggle → *Datasets → New dataset* → upload the zip (name it `erb2-bundle`).
3. New notebook → *Add input* → that dataset; *Settings → Accelerator → GPU T4 x2*; Internet **on**.
4. Run all. Download `/kaggle/working/erb2_results.zip` and unzip into the repo's `results/`.

T4 has no bf16: models load in fp16 with SDPA attention. The run resumes from `results/*.jsonl`, so a
session that hits the 12 h limit can be continued by re-running with the previous results attached.

In [ ]:
import os, glob, shutil, subprocess, zipfile
src = glob.glob("/kaggle/input/*/erb2_bundle.zip") + glob.glob("/kaggle/input/*/*/erb2_bundle.zip")
WORK = "/kaggle/working/erb2"
if src:
    zipfile.ZipFile(src[0]).extractall(WORK)
else:  # dataset uploaded unzipped
    d = [p for p in glob.glob("/kaggle/input/*") if os.path.exists(p + "/src")][0]
    shutil.copytree(d, WORK, dirs_exist_ok=True)
os.chdir(WORK); print(os.listdir())

In [ ]:
!pip install -q -U "transformers>=4.57" accelerate bitsandbytes num2words pyyaml pyarrow
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# T4 = 16 GB: run Qwen2.5-VL-3B unquantised; 7B in 4-bit.
import yaml
cfg = yaml.safe_load(open("configs/verbs.yaml"))
cfg["models"]["qwen2.5-vl-3b"]["quant"] = "none"
yaml.safe_dump(cfg, open("configs/verbs.yaml", "w"), sort_keys=False)
MODELS = ["llava-ov-0.5b", "smolvlm2-2.2b", "internvl3-2b", "qwen2.5-vl-3b"]   # + "qwen2.5-vl-7b" if time allows

In [ ]:
# D3: 50-item smoke test + timing on the smallest model
!python src/04_run_models.py --model llava-ov-0.5b --limit 50 --tag smoke

In [ ]:
for m in MODELS:
    print("=" * 30, m)
    subprocess.run(["python", "src/04_run_models.py", "--model", m], check=False)

In [ ]:
with zipfile.ZipFile("/kaggle/working/erb2_results.zip", "w") as z:
    for f in glob.glob("results/*"):
        if "smoke" not in f:
            z.write(f)
print(os.listdir("results"))